# Vault Notebook Demo

This notebook is a thin UI around `vaultlib`. Set `VAULT_REPO_ROOT`, `VAULT_PATH`, `BACKUP_DIRECTORY`, and `ARCHIVE_PREFIX` in the environment before running it.

In [ ]:
from __future__ import annotations

import os
import sys
from datetime import timezone
from pathlib import Path

import ipywidgets as widgets
from IPython.display import Markdown, display

from src.vaultlib import PythonPassphraseBackend, SecretValue, VaultConfig, VaultService

config = VaultConfig.from_env()
service = VaultService(config, PythonPassphraseBackend())


In [ ]:
def archive_options() -> list[tuple[str, str]]:
    backups = service.list_backups()
    return [(f"{item.filename} ({item.size_bytes} bytes)", str(item.archive_path)) for item in backups]


def refresh_dropdown(dropdown: widgets.Dropdown) -> None:
    options = archive_options()
    dropdown.options = options
    dropdown.disabled = not options
    if not options:
        dropdown.options = [("No backups found", "")]


def format_backup_rows() -> str:
    backups = service.list_backups()
    if not backups:
        return "No backups found."

    lines = ["| Filename | Size | Modified |", "| --- | ---: | --- |"]
    for item in backups:
        modified = item.modified_at.astimezone(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
        lines.append(f"| `{item.filename}` | {item.size_bytes} | {modified} |")
    return "\n".join(lines)


status_output = widgets.Output()
list_output = widgets.Output()


def render_list() -> None:
    with list_output:
        list_output.clear_output()
        display(Markdown(format_backup_rows()))


def render_message(message: str) -> None:
    with status_output:
        status_output.clear_output()
        display(Markdown(message))


backup_name = widgets.Text(description="Filename", placeholder="optional.vaultenc")
backup_passphrase = widgets.Password(description="Passphrase")
backup_overwrite = widgets.Checkbox(description="Allow overwrite", value=False)
backup_button = widgets.Button(description="Create backup", button_style="primary")

verify_archive = widgets.Dropdown(description="Archive")
verify_passphrase = widgets.Password(description="Passphrase")
verify_button = widgets.Button(description="Verify")

restore_archive = widgets.Dropdown(description="Archive")
restore_passphrase = widgets.Password(description="Passphrase")
restore_replace = widgets.Checkbox(description="Replace existing vault", value=False)
restore_button = widgets.Button(description="Restore", button_style="warning")

refresh_button = widgets.Button(description="Refresh backups")


def on_backup(_: widgets.Button) -> None:
    filename = backup_name.value.strip() or None
    try:
        result = service.create_backup(
            SecretValue(backup_passphrase.value),
            archive_name=filename,
            overwrite=backup_overwrite.value,
        )
    except Exception as exc:
        render_message(f"**Backup failed**\n\n```\n{exc}\n```")
        return

    render_message(f"**Backup created**\n\n- Path: `{result.archive_path}`\n- Size: {result.size_bytes} bytes")
    refresh_dropdown(verify_archive)
    refresh_dropdown(restore_archive)
    render_list()


def on_verify(_: widgets.Button) -> None:
    if not verify_archive.value:
        render_message("**Verify failed**\n\nNo archive selected.")
        return
    try:
        result = service.verify_backup(Path(verify_archive.value), SecretValue(verify_passphrase.value))
    except Exception as exc:
        render_message(f"**Verify failed**\n\n```\n{exc}\n```")
        return

    render_message(f"**Verified backup**\n\n- Path: `{result.archive_path}`")


def on_restore(_: widgets.Button) -> None:
    if not restore_archive.value:
        render_message("**Restore failed**\n\nNo archive selected.")
        return
    try:
        result = service.restore_backup(
            Path(restore_archive.value),
            SecretValue(restore_passphrase.value),
            replace_existing=restore_replace.value,
        )
    except Exception as exc:
        render_message(f"**Restore failed**\n\n```\n{exc}\n```")
        return

    recovery_line = (
        f"\n- Recovery copy: `{result.recovered_previous_vault}`" if result.recovered_previous_vault else ""
    )
    render_message(f"**Restored vault**\n\n- Restored to: `{result.restored_to}`{recovery_line}")


def on_refresh(_: widgets.Button) -> None:
    refresh_dropdown(verify_archive)
    refresh_dropdown(restore_archive)
    render_list()
    render_message("**Backup list refreshed**")


backup_button.on_click(on_backup)
verify_button.on_click(on_verify)
restore_button.on_click(on_restore)
refresh_button.on_click(on_refresh)

refresh_dropdown(verify_archive)
refresh_dropdown(restore_archive)
render_list()

display(
    widgets.VBox(
        [
            widgets.HTML("<h3>Backup</h3>"),
            widgets.HBox([backup_name, backup_passphrase]),
            widgets.HBox([backup_overwrite, backup_button]),
            widgets.HTML("<h3>Verify</h3>"),
            widgets.HBox([verify_archive, verify_passphrase, verify_button]),
            widgets.HTML("<h3>Restore</h3>"),
            widgets.HBox([restore_archive, restore_passphrase]),
            widgets.HBox([restore_replace, restore_button]),
            refresh_button,
            widgets.HTML("<h3>Status</h3>"),
            status_output,
            widgets.HTML("<h3>Available Backups</h3>"),
            list_output,
        ]
    )
)
